# SayIt: Training Baseline YOLOv8 Deteksi Uang Rupiah

Notebook ini melatih model baseline dari **dataset publik** yang sudah dirapikan di Roboflow, lalu mengujinya pada **set uji dari foto kalian sendiri**.

**Sebelum mulai:**
1. `Runtime` → `Change runtime type` → pilih **T4 GPU**.
2. Simpan API key Roboflow di **Colab Secrets** (ikon kunci di panel kiri) dengan nama `ROBOFLOW_API_KEY`, lalu aktifkan akses untuk notebook ini. Jangan tulis API key langsung di sel kode.
3. Siapkan dua project di Roboflow:
   - **Data latih**: hasil fork dataset publik yang kelasnya sudah digabung menjadi 7 kelas (`1000`, `2000`, `5000`, `10000`, `20000`, `50000`, `100000`).
   - **Set uji**: foto kalian sendiri, project terpisah, nama kelas **persis sama**.

**Urutan kerja:** jalankan Bagian 1–5 sekarang. Bagian 6–8 dijalankan setelah set uji selesai dilabel.

## 1. Cek GPU dan hubungkan Google Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Semua hasil training disimpan ke Drive agar tidak hilang saat sesi Colab terputus
DRIVE_DIR = Path('/content/drive/MyDrive/sayit')
RUNS_DIR = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
for d in (RUNS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Folder kerja:', DRIVE_DIR)

## 2. Instalasi

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Konfigurasi

Isi nama workspace, project, dan nomor versi sesuai URL Roboflow kalian.
Contoh URL: `https://app.roboflow.com/<workspace>/<project>/<version>`

In [ ]:
# ====== ISI BAGIAN INI ======
TRAIN_WORKSPACE = 'isi-workspace-kalian'
TRAIN_PROJECT   = 'isi-project-data-latih'
TRAIN_VERSION   = 1

TEST_WORKSPACE  = 'isi-workspace-kalian'
TEST_PROJECT    = 'isi-project-set-uji'
TEST_VERSION    = 1

# Urutan kelas yang diharapkan (harus sama dengan hasil penggabungan di Roboflow)
EXPECTED_CLASSES = ['1000', '2000', '5000', '10000', '20000', '50000', '100000']

# Hyperparameter baseline
MODEL_WEIGHTS = 'yolov8n.pt'   # ganti ke 'yolov8s.pt' jika hasil n kurang dan waktu cukup
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16
PATIENCE = 15                  # early stopping jika val tidak membaik
SEED     = 42
RUN_NAME = 'money_baseline_v1'

# Ambang keyakinan untuk evaluasi total nominal (sesuai PRD 8.9)
CONF_THRESHOLD = 0.5
# ============================

In [ ]:
from roboflow import Roboflow

try:
    from google.colab import userdata
    RF_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    from getpass import getpass
    RF_KEY = getpass('Masukkan Roboflow API key: ')

rf = Roboflow(api_key=RF_KEY)

## 4. Unduh dan periksa data latih

In [ ]:
train_ds = (rf.workspace(TRAIN_WORKSPACE)
              .project(TRAIN_PROJECT)
              .version(TRAIN_VERSION)
              .download('yolov8', location='/content/data_train', overwrite=True))
TRAIN_ROOT = Path(train_ds.location)
print('Lokasi data latih:', TRAIN_ROOT)

### 4.1 Perbaiki path di `data.yaml`

File `data.yaml` hasil ekspor Roboflow kadang memakai path relatif yang tidak cocok dengan lokasi di Colab. Sel ini menulis ulang path menjadi absolut dan memastikan nama kelas sesuai harapan.

In [ ]:
import yaml

def find_split_dirs(root: Path):
    """Kembalikan dict split -> folder images yang benar-benar berisi gambar."""
    splits = {}
    for name in ('train', 'valid', 'val', 'test'):
        img_dir = root / name / 'images'
        if img_dir.exists() and any(img_dir.iterdir()):
            splits[name] = img_dir
    return splits

with open(TRAIN_ROOT / 'data.yaml') as f:
    train_cfg = yaml.safe_load(f)

train_names = train_cfg['names']
if isinstance(train_names, dict):
    train_names = [train_names[i] for i in sorted(train_names)]
train_names = [str(n) for n in train_names]
print('Kelas di dataset:', train_names)

missing = set(EXPECTED_CLASSES) - set(train_names)
extra = set(train_names) - set(EXPECTED_CLASSES)
assert not missing and not extra, (
    f'Kelas belum rapi. Hilang: {missing} | Kelebihan: {extra}. '
    'Perbaiki di Roboflow (ganti nama/gabung/hapus kelas), buat versi baru, lalu ulangi.'
)

splits = find_split_dirs(TRAIN_ROOT)
print('Split ditemukan:', {k: str(v) for k, v in splits.items()})
assert 'train' in splits, 'Split train tidak ditemukan.'
val_key = 'valid' if 'valid' in splits else ('val' if 'val' in splits else None)
assert val_key, 'Split validasi tidak ditemukan. Atur pembagian train/valid/test di Roboflow.'

train_yaml = {
    'path': str(TRAIN_ROOT),
    'train': str(splits['train']),
    'val': str(splits[val_key]),
    'names': {i: n for i, n in enumerate(train_names)},
}
if 'test' in splits:
    train_yaml['test'] = str(splits['test'])

TRAIN_YAML = TRAIN_ROOT / 'data_fixed.yaml'
with open(TRAIN_YAML, 'w') as f:
    yaml.safe_dump(train_yaml, f, sort_keys=False)
print(open(TRAIN_YAML).read())

### 4.2 Statistik data

Periksa jumlah gambar per split dan jumlah objek per kelas. Kelas yang jumlahnya jauh lebih sedikit biasanya menjadi kelas dengan performa terburuk.

In [ ]:
from collections import Counter
import pandas as pd

def label_dir_for(img_dir: Path) -> Path:
    return img_dir.parent / 'labels'

def class_counts(img_dir: Path, names):
    counts = Counter()
    empty = 0
    for lbl in label_dir_for(img_dir).glob('*.txt'):
        lines = [l for l in lbl.read_text().splitlines() if l.strip()]
        if not lines:
            empty += 1
        for l in lines:
            counts[names[int(l.split()[0])]] += 1
    return counts, empty

rows = []
for split, img_dir in splits.items():
    n_img = len([p for p in img_dir.iterdir() if p.is_file()])
    counts, empty = class_counts(img_dir, train_names)
    row = {'split': split, 'gambar': n_img, 'tanpa_objek': empty}
    row.update({c: counts.get(c, 0) for c in train_names})
    rows.append(row)

stats = pd.DataFrame(rows).set_index('split')
display(stats)

### 4.3 Cek duplikat antar split

Gambar yang persis sama di train dan valid/test membuat skor validasi terlihat lebih bagus dari kenyataan. Cek ini hanya menangkap duplikat identik (hash sama), bukan foto yang mirip.

In [ ]:
import hashlib

def md5(p: Path):
    return hashlib.md5(p.read_bytes()).hexdigest()

hashes = {split: {md5(p) for p in d.iterdir() if p.is_file()} for split, d in splits.items()}
split_names = list(hashes)
found = False
for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        a, b = split_names[i], split_names[j]
        dup = hashes[a] & hashes[b]
        if dup:
            found = True
            print(f'PERINGATAN: {len(dup)} gambar identik muncul di {a} dan {b}')
if not found:
    print('Tidak ada duplikat identik antar split.')

## 5. Training baseline

Hasil disimpan langsung ke Google Drive (`sayit/runs/...`). Checkpoint disimpan setiap 5 epoch.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    save_period=5,
    plots=True,
)
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
print('Bobot terbaik:', BEST_PT)

### 5.1 Jika sesi terputus

Jalankan Bagian 1–3 lagi, lalu jalankan sel di bawah ini (hapus tanda `#`) untuk melanjutkan dari checkpoint terakhir.

In [ ]:
# RUN_DIR = RUNS_DIR / RUN_NAME
# model = YOLO(str(RUN_DIR / 'weights' / 'last.pt'))
# model.train(resume=True)
# BEST_PT = RUN_DIR / 'weights' / 'best.pt'

### 5.2 Hasil pada split validasi (data publik)

Angka ini **bukan** ukuran performa sebenarnya, karena gambar validasi berasal dari sumber yang sama dengan data latih. Anggap sebagai pemeriksaan bahwa training berjalan wajar.

In [ ]:
from IPython.display import Image as IPImage, display

model = YOLO(str(BEST_PT))
val_metrics = model.val(data=str(TRAIN_YAML), split='val', project=str(RUNS_DIR),
                        name=RUN_NAME + '_val', exist_ok=True)
print(f'Validasi publik  mAP@0.5 = {val_metrics.box.map50:.3f} | mAP@0.5:0.95 = {val_metrics.box.map:.3f}')

for fname in ('results.png', 'confusion_matrix.png'):
    p = RUN_DIR / fname
    if p.exists():
        display(IPImage(filename=str(p), width=800))

## 6. Siapkan set uji

Pilih **salah satu** jalur:
- **6a** (sel di bawah): set uji dari project Roboflow terpisah, baik foto sendiri maupun dataset publik lain.
- **6b**: memakai split `test` dari data latih (paling cepat, tetapi paling tidak jujur).

Setelah itu jalankan **6.1** (hanya untuk dataset publik lain) dan **6.2** (selalu).

### 6a. Unduh set uji dari project Roboflow terpisah

Sel ini **menyamakan urutan kelas** set uji dengan data latih. Tanpa langkah ini, kelas bisa tertukar diam-diam (misalnya indeks 0 di set uji adalah `10000`, tetapi di model adalah `1000`) dan hasil evaluasi menjadi salah total.

In [ ]:
import shutil

test_ds = (rf.workspace(TEST_WORKSPACE)
             .project(TEST_PROJECT)
             .version(TEST_VERSION)
             .download('yolov8', location='/content/data_test_raw', overwrite=True))
TEST_RAW = Path(test_ds.location)

with open(TEST_RAW / 'data.yaml') as f:
    test_cfg = yaml.safe_load(f)
test_names = test_cfg['names']
if isinstance(test_names, dict):
    test_names = [test_names[i] for i in sorted(test_names)]
test_names = [str(n) for n in test_names]
print('Kelas set uji:', test_names)

unknown = set(test_names) - set(train_names)
assert not unknown, f'Kelas set uji tidak dikenal model: {unknown}. Samakan nama kelas di Roboflow.'

remap = {i: train_names.index(n) for i, n in enumerate(test_names)}
print('Pemetaan indeks uji -> model:', remap)

# Gabungkan semua split set uji ke satu folder, lalu tulis ulang label dengan indeks model
TEST_ROOT = Path('/content/data_test')
if TEST_ROOT.exists():
    shutil.rmtree(TEST_ROOT)
(TEST_ROOT / 'images').mkdir(parents=True)
(TEST_ROOT / 'labels').mkdir(parents=True)

n_copied = 0
for split, img_dir in find_split_dirs(TEST_RAW).items():
    for img in img_dir.iterdir():
        if not img.is_file():
            continue
        new_stem = f'{split}_{img.stem}'
        shutil.copy(img, TEST_ROOT / 'images' / f'{new_stem}{img.suffix}')
        src_lbl = label_dir_for(img_dir) / f'{img.stem}.txt'
        lines = []
        if src_lbl.exists():
            for l in src_lbl.read_text().splitlines():
                if l.strip():
                    parts = l.split()
                    parts[0] = str(remap[int(parts[0])])
                    lines.append(' '.join(parts))
        (TEST_ROOT / 'labels' / f'{new_stem}.txt').write_text('\n'.join(lines))
        n_copied += 1

TEST_YAML = TEST_ROOT / 'data.yaml'
with open(TEST_YAML, 'w') as f:
    yaml.safe_dump({
        'path': str(TEST_ROOT),
        'train': str(TEST_ROOT / 'images'),  # tidak dipakai, hanya pelengkap format
        'val': str(TEST_ROOT / 'images'),
        'test': str(TEST_ROOT / 'images'),
        'names': {i: n for i, n in enumerate(train_names)},
    }, f, sort_keys=False)

print(f'{n_copied} gambar uji siap di {TEST_ROOT}')
test_counts, test_empty = class_counts(TEST_ROOT / 'images', train_names)
display(pd.DataFrame([{**{c: test_counts.get(c, 0) for c in train_names}, 'tanpa_objek': test_empty}],
                     index=['set_uji']))

### 6b. Alternatif: split test dari data latih

Lewati sel 6a jika memakai jalur ini.

In [ ]:
# assert 'test' in splits, 'Split test tidak ada di dataset latih.'
# TEST_ROOT = splits['test'].parent      # berisi folder images/ dan labels/
# TEST_YAML = TRAIN_YAML                 # indeks kelas sudah sama, tidak perlu remap
# print('Set uji:', TEST_ROOT, '|', len(list((TEST_ROOT / 'images').iterdir())), 'gambar')

### 6.1 Buang gambar uji yang mirip data latih (khusus dataset publik lain)

Dua dataset publik bisa berisi foto yang sama. Cek duplikat di 4.3 tidak menangkapnya karena Roboflow mengompres ulang gambar, jadi sel ini memakai *perceptual hash*. Lewati sel ini untuk jalur 6b dan untuk foto sendiri.

In [ ]:
!pip install -q imagehash
import numpy as np
import imagehash
from PIL import Image

def phash_bits(p: Path):
    with Image.open(p) as im:
        return imagehash.phash(im.convert('RGB')).hash.flatten()

train_imgs = [p for d in splits.values() for p in d.iterdir() if p.is_file()]
test_imgs = sorted(p for p in (TEST_ROOT / 'images').iterdir() if p.is_file())
H_train = np.stack([phash_bits(p) for p in train_imgs])
H_test = np.stack([phash_bits(p) for p in test_imgs])

MAX_DIST = 8   # jarak Hamming maksimum (dari 64 bit) untuk dianggap mirip
removed = []
for i in range(0, len(H_test), 200):
    chunk = H_test[i:i + 200]
    dist = (chunk[:, None, :] != H_train[None, :, :]).sum(-1).min(axis=1)
    for k, dmin in enumerate(dist):
        if dmin <= MAX_DIST:
            img = test_imgs[i + k]
            removed.append((img.name, int(dmin)))
            img.unlink()
            (TEST_ROOT / 'labels' / f'{img.stem}.txt').unlink(missing_ok=True)

print(f'{len(removed)} dari {len(test_imgs)} gambar uji dibuang karena mirip data latih.')
print(f'Sisa set uji: {len(test_imgs) - len(removed)} gambar')
for name, d in removed[:10]:
    print(f'  {name}  (jarak {d})')

### 6.2 Normalisasi label set uji

Dua masalah label yang ditemukan pada dataset publik:
1. **Satu file label berisi campuran kotak dan poligon.** Ultralytics mengabaikan seluruh gambar seperti ini ("ignoring corrupt image/label"), sehingga gambar tersebut tidak ikut dievaluasi.
2. **Satu lembar uang diberi dua anotasi** dengan kelas sama (misalnya karena kelas "Seribu" dan "seribu rupiah" digabung). Anotasi ganda membuat recall turun dan jumlah lembar di 7.2 terhitung dobel.

Sel ini mengubah semua poligon menjadi kotak pembungkus dan membuang kotak ganda (kelas sama, IoU > `DUP_IOU`).

In [ ]:
import numpy as np

DUP_IOU = 0.7

def xywh_to_xyxy(a):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2,
                     a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], axis=1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def normalize_labels(lbl_dir: Path, dup_iou: float = DUP_IOU):
    n_files = n_poly = n_dup = n_changed = 0
    for lbl in sorted(lbl_dir.glob('*.txt')):
        n_files += 1
        cls, boxes, had_poly = [], [], False
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            vals = list(map(float, parts[1:]))
            if len(vals) != 4:
                xs, ys = vals[0::2], vals[1::2]
                vals = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
                        max(xs) - min(xs), max(ys) - min(ys)]
                n_poly += 1
                had_poly = True
            cls.append(int(parts[0]))
            boxes.append(vals)
        xyxy = xywh_to_xyxy(boxes) if boxes else np.zeros((0, 4))
        keep = []
        for i in range(len(cls)):
            is_dup = any(cls[j] == cls[i] and iou_matrix(xyxy[i:i + 1], xyxy[j:j + 1])[0, 0] > dup_iou
                         for j in keep)
            if is_dup:
                n_dup += 1
            else:
                keep.append(i)
        if had_poly or len(keep) != len(cls):
            n_changed += 1
            lbl.write_text('\n'.join(
                f'{cls[i]} ' + ' '.join(f'{v:.6f}' for v in boxes[i]) for i in keep))
    # hapus cache Ultralytics agar label baru terbaca
    (lbl_dir.parent / 'labels.cache').unlink(missing_ok=True)
    print(f'{n_files} file diperiksa | {n_changed} file diubah | '
          f'{n_poly} poligon -> kotak | {n_dup} anotasi ganda dibuang')

normalize_labels(TEST_ROOT / 'labels')
test_counts, test_empty = class_counts(TEST_ROOT / 'images', train_names)
n_imgs = len([p for p in (TEST_ROOT / 'images').iterdir() if p.is_file()])
display(pd.DataFrame([{'gambar': n_imgs, **{c: test_counts.get(c, 0) for c in train_names},
                       'tanpa_objek': test_empty}], index=['set_uji_bersih']))

## 7. Evaluasi pada set uji

Angka yang dilaporkan sebagai performa model adalah hasil pada **foto sendiri**. Hasil pada dataset publik ditulis sebagai evaluasi pada data publik.

### 7.1 Metrik deteksi

In [ ]:
test_metrics = model.val(data=str(TEST_YAML), split='test', project=str(RUNS_DIR),
                         name=RUN_NAME + '_test', exist_ok=True)
print(f'SET UJI  mAP@0.5 = {test_metrics.box.map50:.3f} | mAP@0.5:0.95 = {test_metrics.box.map:.3f}')
print(f'Penurunan dari validasi publik: {val_metrics.box.map50 - test_metrics.box.map50:+.3f} (mAP@0.5)')

# Tabel per kelas
try:
    idx = list(test_metrics.box.ap_class_index)
    per_class = pd.DataFrame({
        'kelas': [train_names[i] for i in idx],
        'precision': test_metrics.box.p,
        'recall': test_metrics.box.r,
        'AP@0.5': test_metrics.box.ap50,
        'AP@0.5:0.95': test_metrics.box.ap,
    }).sort_values('AP@0.5')
    display(per_class.round(3))
except Exception as e:
    print('Tabel per kelas gagal dibuat, cek versi ultralytics:', e)

test_dir = Path(getattr(test_metrics, 'save_dir', RUNS_DIR / (RUN_NAME + '_test')))
cm = test_dir / 'confusion_matrix.png'
if cm.exists():
    display(IPImage(filename=str(cm), width=800))

### 7.2 Akurasi total nominal per foto

Metrik tingkat sistem: apakah total uang yang dihitung model sama persis dengan total sebenarnya. Hanya deteksi dengan skor ≥ `CONF_THRESHOLD` yang dihitung, sesuai PRD.

In [ ]:
NOMINAL = {name: int(name) for name in train_names}

def true_total(lbl_path: Path):
    total, count = 0, 0
    if lbl_path.exists():
        for l in lbl_path.read_text().splitlines():
            if l.strip():
                total += NOMINAL[train_names[int(l.split()[0])]]
                count += 1
    return total, count

records = []
img_paths = sorted(p for p in (TEST_ROOT / 'images').iterdir() if p.is_file())
for img in img_paths:
    res = model.predict(source=str(img), conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)[0]
    pred_cls = [int(c) for c in res.boxes.cls.tolist()]
    pred_total = sum(NOMINAL[train_names[c]] for c in pred_cls)
    gt_total, gt_count = true_total(TEST_ROOT / 'labels' / f'{img.stem}.txt')
    records.append({
        'file': img.name,
        'total_benar': gt_total,
        'total_prediksi': pred_total,
        'jumlah_lembar_benar': gt_count,
        'jumlah_lembar_prediksi': len(pred_cls),
        'total_tepat': gt_total == pred_total,
        'jumlah_tepat': gt_count == len(pred_cls),
    })

eval_df = pd.DataFrame(records)
print(f'Akurasi total nominal : {eval_df.total_tepat.mean():.1%}  ({eval_df.total_tepat.sum()}/{len(eval_df)})')
print(f'Akurasi jumlah lembar : {eval_df.jumlah_tepat.mean():.1%}')
print('\nAkurasi per jumlah lembar di foto:')
display(eval_df.groupby('jumlah_lembar_benar').total_tepat.agg(['mean', 'count']).round(3))

eval_csv = RUNS_DIR / f'{RUN_NAME}_test_totals.csv'
eval_df.to_csv(eval_csv, index=False)
print('Disimpan ke', eval_csv)

### 7.3 Galeri kesalahan

Lihat foto yang salah hitung. Catat pola kegagalannya (gelap, terlipat, emisi tertentu, tertutup jari) karena pola ini yang menentukan foto latih tambahan apa yang perlu diambil.

In [ ]:
import matplotlib.pyplot as plt

wrong = eval_df[~eval_df.total_tepat].head(12)
if wrong.empty:
    print('Tidak ada kesalahan total nominal pada set uji.')
else:
    cols = 3
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, (_, r) in zip(axes, wrong.iterrows()):
        res = model.predict(source=str(TEST_ROOT / 'images' / r.file), conf=CONF_THRESHOLD,
                            imgsz=IMGSZ, verbose=False)[0]
        ax.imshow(res.plot()[:, :, ::-1])
        ax.set_title(f'{r.file[:25]}\nbenar {r.total_benar:,} | prediksi {r.total_prediksi:,}', fontsize=9)
        ax.axis('off')
    for ax in axes[len(wrong):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### 7.4 Audit label dengan bantuan model

Sel ini mencari objek yang **labelnya berbeda dari prediksi model** padahal model sangat yakin (skor ≥ `AUDIT_CONF`) dan kotaknya bertumpuk (IoU ≥ 0,5). Kasus seperti ini sering berarti **labelnya yang salah**, bukan modelnya.

Yang diperiksa adalah seluruh split data latih (train, valid, test). Keterbatasan: model sudah "menghafal" label train, jadi label salah di train tidak semuanya akan muncul. Hasilnya adalah daftar kandidat untuk dicek manual di Roboflow, bukan kepastian.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

AUDIT_CONF = 0.70
AUDIT_IOU = 0.5

def xywh_to_xyxy(a):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2,
                     a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], axis=1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def read_gt(lbl_path: Path):
    """Baca label YOLO. Anotasi poligon diubah menjadi kotak pembungkus."""
    cls, boxes = [], []
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            vals = list(map(float, parts[1:]))
            if len(vals) == 4:
                box = vals
            else:
                xs, ys = vals[0::2], vals[1::2]
                box = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
                       max(xs) - min(xs), max(ys) - min(ys)]
            cls.append(int(parts[0]))
            boxes.append(box)
    return cls, (xywh_to_xyxy(boxes) if boxes else np.zeros((0, 4)))

audit = []
for split, img_dir in splits.items():
    for img in sorted(p for p in img_dir.iterdir() if p.is_file()):
        gt_cls, gt_box = read_gt(label_dir_for(img_dir) / f'{img.stem}.txt')
        if not gt_cls:
            continue
        res = model.predict(source=str(img), conf=AUDIT_CONF, imgsz=IMGSZ, verbose=False)[0]
        if len(res.boxes) == 0:
            continue
        pr_cls = res.boxes.cls.cpu().numpy().astype(int)
        pr_conf = res.boxes.conf.cpu().numpy()
        pr_box = xywh_to_xyxy(res.boxes.xywhn.cpu().numpy())
        ious = iou_matrix(gt_box, pr_box)
        for gi, gc in enumerate(gt_cls):
            pj = int(ious[gi].argmax())
            if ious[gi, pj] >= AUDIT_IOU and pr_cls[pj] != gc:
                audit.append({
                    'split': split,
                    'file': img.name,
                    'nama_asli_kira2': img.name.split('.rf.')[0],
                    'label': train_names[gc],
                    'prediksi': train_names[pr_cls[pj]],
                    'conf': round(float(pr_conf[pj]), 2),
                    'iou': round(float(ious[gi, pj]), 2),
                })

audit_df = pd.DataFrame(audit)
if audit_df.empty:
    print('Tidak ada kandidat label salah pada ambang ini.')
else:
    print(f'{len(audit_df)} kandidat label salah di {audit_df.file.nunique()} gambar\n')
    print('Pola label -> prediksi:')
    display(audit_df.groupby(['label', 'prediksi']).size()
            .sort_values(ascending=False).rename('jumlah').to_frame())
    display(audit_df.sort_values('conf', ascending=False))
    audit_csv = RUNS_DIR / f'{RUN_NAME}_audit_label.csv'
    audit_df.to_csv(audit_csv, index=False)
    print('Disimpan ke', audit_csv)

In [ ]:
# Galeri kandidat: kotak hijau = label saat ini, judul = prediksi model
if not audit_df.empty:
    show = audit_df.drop_duplicates('file').head(12)
    cols = 3
    rows = (len(show) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.8 * rows))
    axes = np.atleast_1d(axes).flatten()
    for ax, (_, r) in zip(axes, show.iterrows()):
        img_path = splits[r.split] / r.file
        im = cv2.imread(str(img_path))
        h, w = im.shape[:2]
        gt_cls, gt_box = read_gt(label_dir_for(splits[r.split]) / f'{img_path.stem}.txt')
        for c, (x1, y1, x2, y2) in zip(gt_cls, gt_box):
            p1, p2 = (int(x1 * w), int(y1 * h)), (int(x2 * w), int(y2 * h))
            cv2.rectangle(im, p1, p2, (0, 200, 0), 3)
            cv2.putText(im, train_names[c], (p1[0] + 4, p1[1] + 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 0), 2)
        ax.imshow(im[:, :, ::-1])
        wrong = audit_df[audit_df.file == r.file]
        info = ', '.join(f'{a}->{b} ({c})' for a, b, c in zip(wrong.label, wrong.prediksi, wrong.conf))
        ax.set_title(f'[{r.split}] {r.nama_asli_kira2[:28]}\nlabel->prediksi: {info}', fontsize=8)
        ax.axis('off')
    for ax in axes[len(show):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 8. Keputusan dan penyimpanan model

Patokan awal (sesuaikan dengan target yang disepakati tim):
- **mAP@0.5 set uji ≥ 0,80 dan akurasi total ≥ 85%**: baseline cukup. Lanjut ke modul lain.
- **Di bawah itu**: tambahkan foto latih **hanya** untuk kelas dan kondisi yang paling sering gagal (lihat tabel 7.1 dan galeri 7.3), lalu latih ulang dengan `RUN_NAME` baru.

Jangan pernah memasukkan foto set uji ke data latih. Jika set uji ikut dilatih, angka evaluasi tidak lagi bisa dipercaya.

In [ ]:
import shutil
from datetime import datetime

map50 = test_metrics.box.map50
acc_total = eval_df.total_tepat.mean()
print(f'mAP@0.5 set uji     : {map50:.3f}')
print(f'Akurasi total nominal: {acc_total:.1%}')

if map50 >= 0.80 and acc_total >= 0.85:
    print('\nKEPUTUSAN: baseline memadai. Lanjut ke modul pipeline.')
else:
    print('\nKEPUTUSAN: perlu data latih tambahan. Kelas terlemah:')
    try:
        display(per_class.head(3))
    except NameError:
        pass

stamp = datetime.now().strftime('%Y%m%d_%H%M')
dst = MODELS_DIR / f'best_money_{RUN_NAME}_{stamp}.pt'
shutil.copy(BEST_PT, dst)
print('\nModel disalin ke', dst)